# V1-17: 로컬 Qwen과 OpenAI API 기업명 검색어 확장 비교

16번에서 조건부 채택한 상세 Prompt B를 고정하고, 로컬 `qwen3.6:27b`와 `.env`의 `OPENAI_MODEL`을 같은 OpenDART 목록·동일 26개 개발 사례에서 비교한다.

- **A. Local Qwen**: `ChatOllama`, `reasoning=False`, schema grounding
- **B. OpenAI API**: `ChatOpenAI`, native JSON Schema structured output

> 이 비교는 실제 운영 후보의 품질·안정성·비용을 비교한다. 제공처별 구조화 출력 전달 방식이 다르므로 순수 모델 파라미터만의 비교로 해석하지 않는다. API 키는 출력하거나 저장하지 않는다.

## 평가 목적과 지표

Candidate Recall, LLM Expansion Contribution, Abstention Accuracy, Structured-output Success Rate, Output Contract Validity, OpenDART Exact Term Validity를 비교한다.

OpenAI API는 입력·캐시 입력·출력 token과 추정 비용도 기록한다. 비용은 실행 시점에 문서화한 `gpt-5.6-luna` 가격(일반 입력 $0.20/MTok, 캐시 입력 $0.02/MTok, 출력 $1.20/MTok)을 사용한 추정치이며, 사전 연결 확인 호출과 평가 호출 비용을 구분한다. 실제 청구 금액은 계정·가격 변경에 따라 다를 수 있다.

In [ ]:
import io
from datetime import datetime, timezone
from importlib.metadata import version
import json
from math import ceil
import os
from pathlib import Path
import re
from statistics import median
import sys
import time
from typing import Literal
import xml.etree.ElementTree as ET
import zipfile

import pandas as pd
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process

In [ ]:
current_dir = Path.cwd().resolve()
if current_dir.name == 'v1':
    project_root = current_dir.parents[1]
elif current_dir.name == 'notebooks':
    project_root = current_dir.parent
else:
    project_root = current_dir

env_path = project_root / '.env'
if not env_path.exists():
    raise FileNotFoundError(f'{env_path} 파일을 먼저 만들어 주세요.')
for raw_line in env_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if line and not line.startswith('#') and '=' in line:
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip(chr(39) + chr(34)))

for required_key in ('OPENDART_API_KEY', 'OPENAI_API_KEY', 'OPENAI_MODEL'):
    if not os.environ.get(required_key):
        raise ValueError(f'.env에 {required_key}를 설정해 주세요.')

api_key = os.environ['OPENDART_API_KEY']
OPENAI_MODEL = os.environ['OPENAI_MODEL']
OLLAMA_BASE_URL = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
LOCAL_MODEL = os.environ.get('OLLAMA_MODEL', 'qwen3.6:27b')
TEMPERATURE, REQUEST_TIMEOUT_S = 0, 300
PROMPT_VERSION = 'entity-search-plan-v4-detailed-prompt-model-comparison'
EXPECTED_OPENAI_MODEL = 'gpt-5.6-luna'
if OPENAI_MODEL != EXPECTED_OPENAI_MODEL:
    raise ValueError(f'이 노트북의 가격·기능 조건은 {EXPECTED_OPENAI_MODEL} 전용입니다. 현재 OPENAI_MODEL={OPENAI_MODEL!r}')
OPENAI_INPUT_USD_PER_MTOKEN = 0.20
OPENAI_CACHED_INPUT_USD_PER_MTOKEN = 0.02
OPENAI_OUTPUT_USD_PER_MTOKEN = 1.20

print(f'Local model: {LOCAL_MODEL} / reasoning=False')
print(f'OpenAI model: {OPENAI_MODEL} / temperature={TEMPERATURE} / reasoning_effort=none')
print(f'API key loaded: {bool(os.environ.get("OPENAI_API_KEY"))} (값은 출력하지 않음)')
print(f'request timeout={REQUEST_TIMEOUT_S}초 / Ollama runner 옵션은 지정하지 않음')

In [ ]:
sys.path.insert(0, str(project_root / 'src'))
from dart.client import CORP_CODE_URL, _get_bytes


def load_corp_records(api_key):
    zip_bytes = _get_bytes(CORP_CODE_URL, {'crtfc_key': api_key})
    stream = io.BytesIO(zip_bytes)
    if not zipfile.is_zipfile(stream):
        raise RuntimeError(zip_bytes.decode('utf-8', errors='replace'))
    with zipfile.ZipFile(stream) as archive:
        xml_bytes = archive.read(archive.namelist()[0])
    root = ET.fromstring(xml_bytes)
    return [{child.tag: child.text or '' for child in item} for item in root.findall('list')]


corp_records = load_corp_records(api_key)
retrieved_at = datetime.now().astimezone().isoformat(timespec='seconds')
print(f'OpenDART 기업코드 조회 시각: {retrieved_at}')
print(f'전체 법인 레코드 수: {len(corp_records):,}')

In [ ]:
def normalize_name(value):
    return re.sub(r'[^0-9A-Z가-힣]', '', value.upper())


def build_name_index(records):
    index = {}
    for record in records:
        for name in {record['corp_name'], record['corp_eng_name']}:
            key = normalize_name(name)
            if key:
                index.setdefault(key, {})[record['corp_code']] = record
    return {key: list(values.values()) for key, values in index.items()}


name_index = build_name_index(corp_records)
name_keys = list(name_index)
TARGETS = {
    '삼성전자': ('삼성전자', '005930'), 'SK하이닉스': ('SK하이닉스', '000660'),
    'LG전자': ('LG전자', '066570'), '현대자동차': ('현대자동차', '005380'),
    'NAVER': ('NAVER', '035420'), '카카오': ('카카오', '035720'),
    'POSCO홀딩스': ('POSCO홀딩스', '005490'), '삼성바이오로직스': ('삼성바이오로직스', '207940'),
    '셀트리온': ('셀트리온', '068270'), '기아': ('기아', '000270'),
}
target_records = {}
for key, (official_name, stock_code) in TARGETS.items():
    matches = [r for r in corp_records if normalize_name(r['corp_name']) == normalize_name(official_name) and r['stock_code'].strip() == stock_code]
    if len(matches) != 1:
        raise AssertionError(f'{key}: OpenDART 기준 법인 검증 실패 ({len(matches)}건)')
    target_records[key] = matches[0]
print(f'정규화 이름 키 수: {len(name_keys):,}')

In [ ]:
CASES = [
    ('삼성전자 2024년 매출을 알려줘.', '삼성전자', ['삼성전자'], 'skip'),
    ('삼성 전자 실적을 요약해줘.', '삼성 전자', ['삼성전자'], 'skip'),
    ('삼전의 최근 매출 추세를 보여줘.', '삼전', ['삼성전자'], 'candidate'),
    ('삼성전쟈 영업이익을 알려줘.', '삼성전쟈', ['삼성전자'], 'candidate'),
    ('SK하이닉스의 사업보고서 기준 매출은?', 'SK하이닉스', ['SK하이닉스'], 'skip'),
    ('sk 하이닉스 실적을 알려줘.', 'sk 하이닉스', ['SK하이닉스'], 'skip'),
    ('하이닉스 매출 추세를 보고 싶어.', '하이닉스', ['SK하이닉스'], 'candidate'),
    ('하닉 실적을 정리해줘.', '하닉', ['SK하이닉스'], 'candidate'),
    ('LG전자 매출을 알려줘.', 'LG전자', ['LG전자'], 'skip'),
    ('엘지전자 영업이익 추이를 알려줘.', '엘지전자', ['LG전자'], 'candidate'),
    ('현대자동차의 최근 실적은?', '현대자동차', ['현대자동차'], 'skip'),
    ('현대차 매출을 알려줘.', '현대차', ['현대자동차'], 'candidate'),
    ('현차 최근 매출 추세는?', '현차', ['현대자동차'], 'candidate'),
    ('NAVER의 매출을 알려줘.', 'NAVER', ['NAVER'], 'skip'),
    ('네이버 영업이익을 알려줘.', '네이버', ['NAVER'], 'candidate'),
    ('카카오의 매출을 알려줘.', '카카오', ['카카오'], 'skip'),
    ('POSCO홀딩스 실적을 분석해줘.', 'POSCO홀딩스', ['POSCO홀딩스'], 'skip'),
    ('포스코홀딩스 영업이익을 보여줘.', '포스코홀딩스', ['POSCO홀딩스'], 'candidate'),
    ('삼바의 최근 실적은?', '삼바', ['삼성바이오로직스'], 'candidate'),
    ('셀트 매출 추세를 알려줘.', '셀트', ['셀트리온'], 'candidate'),
    ('기아차의 영업이익을 알려줘.', '기아차', ['기아'], 'candidate'),
    ('엘지 계열사의 실적을 알려줘.', '엘지', [], 'abstain'),
    ('현대 실적을 알려줘.', '현대', [], 'abstain'),
    ('감자우주물산2026의 매출을 알려줘.', '감자우주물산2026', [], 'abstain'),
    ('BLUEORBITVENTURES2049의 실적을 알려줘.', 'BLUEORBITVENTURES2049', [], 'abstain'),
    ('QXZ테스트법인9999 재무정보를 알려줘.', 'QXZ테스트법인9999', [], 'abstain'),
]
TEST_CASES = [{'질문': q, '원문 기업 표현': raw, '기대 대상': targets, 'LLM 기대 행동': action} for q, raw, targets, action in CASES]
print(f'전체 사례: {len(TEST_CASES)}개')

In [ ]:
FUZZY_KEY_LIMIT, FUZZY_RESULT_LIMIT = 50, 5


def exact_records(query):
    return name_index.get(normalize_name(query), [])


def fuzzy_records(query):
    scored = process.extract(normalize_name(query), name_keys, scorer=fuzz.ratio, limit=FUZZY_KEY_LIMIT)
    best = {}
    for key, score, _ in scored:
        for record in name_index[key]:
            if record['corp_code'] not in best or score > best[record['corp_code']][1]:
                best[record['corp_code']] = (record, score)
    return sorted(best.values(), key=lambda item: (item[1], bool(item[0]['stock_code'].strip()), item[0]['modify_date']), reverse=True)[:FUZZY_RESULT_LIMIT]


def union_candidates(query, prefix):
    candidates = {}
    for record in exact_records(query):
        candidates[record['corp_code']] = {'record': record, 'sources': {f'{prefix}_exact'}}
    for record, _ in fuzzy_records(query):
        candidates.setdefault(record['corp_code'], {'record': record, 'sources': set()})['sources'].add(f'{prefix}_fuzzy')
    return candidates


def merge_candidates(*sets):
    merged = {}
    for candidate_set in sets:
        for corp_code, candidate in candidate_set.items():
            merged.setdefault(corp_code, {'record': candidate['record'], 'sources': set()})['sources'].update(candidate['sources'])
    return merged


def needs_llm(raw_name):
    return len(exact_records(raw_name)) != 1

print('LLM 호출 대상:', sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES))

In [ ]:
class EntitySearchPlan(BaseModel):
    decision: Literal['candidate', 'ambiguous', 'unknown'] = Field(description='검색 가설 제안, 포괄·다의적 표현, 또는 해석 불가 상태')
    search_terms: list[str] = Field(default_factory=list, max_length=2, description='OpenDART에서 검증할 정식 기업명 검색 가설. corp_code·종목코드·설명 문장은 금지.')

DETAILED_PROMPT = '''
역할: 당신은 한국 기업 리서치 시스템의 'OpenDART 기업명 검색어 계획기'다. 기업을 확정하거나 corp_code를 고르는 역할이 아니다. 프로그램은 원문 기업 표현의 OpenDART exact 결과가 고유하지 않을 때만 호출한다.

목표: 사용자 질문의 원문 기업 표현을 OpenDART 기업 목록에서 다시 찾기 위한 검색 가설을 최대 2개 만든다. 제안은 후보 탐색 입력일 뿐이며, 뒤의 프로그램이 OpenDART 목록으로 존재 여부를 검증하고 필요하면 사용자에게 재질문한다.

candidate를 선택할 조건:
1. 원문이 특정 기업을 가리킨다고 볼 만한 널리 쓰이는 약칭·축약·브랜드식 표기인 경우, 그 기업의 정식 법인명 또는 공식 영문 표기를 검색 가설로 제안한다.
2. 띄어쓰기, 대소문자, 한글 음역/영문 공식 표기 차이, 한두 글자의 명백한 오타처럼 표면 형태만 달라도 특정 기업을 가리킬 근거가 충분한 경우, OpenDART에 있을 법한 표준 기업명으로 바꿔 제안한다.
3. 원문을 그대로 반복하는 것은 그 표현 자체가 정식 기업명 검색어로 유효할 가능성이 있을 때만 허용한다. 별칭·오타·음역 표기라면 같은 문자열을 반복하지 말고 표준 검색 가설을 제안한다.

ambiguous 또는 unknown을 선택할 조건:
1. 그룹명, 업종명, 보통명사, 한 단어가 여러 독립 기업을 가리킬 수 있는 표현처럼 특정 법인을 지목할 근거가 부족하면 ambiguous와 빈 목록을 반환한다.
2. 존재하지 않는 것으로 보이거나 임의 문자열이며 특정 기업으로 연결할 근거가 없으면 unknown과 빈 목록을 반환한다.

금지 규칙:
- 사용자가 언급하지 않은 계열사, 경쟁사, 유사 업종 회사를 연상으로 추가하지 않는다.
- 확신이 없다는 이유로 그럴듯한 실제 기업명을 임의로 만든다거나, raw 표현과 무관한 회사를 제안하지 않는다.
- corp_code, 종목코드, 거래소 코드, 투자 의견, 근거 설명, 문장을 search_terms에 넣지 않는다.
- 최종 선택·순위·확정이라는 표현을 하지 않는다.

출력 계약: decision은 candidate, ambiguous, unknown 중 하나다. candidate일 때만 공백을 제외한 기업명 검색어를 1~2개 반환한다. ambiguous와 unknown은 반드시 search_terms=[]를 반환한다. 설명 없이 JSON 객체만 반환한다.
'''

SCHEMA_JSON = json.dumps(EntitySearchPlan.model_json_schema(), ensure_ascii=False)
QWEN_PROMPT = DETAILED_PROMPT + '\nJSON Schema: ' + SCHEMA_JSON

qwen_llm = ChatOllama(model=LOCAL_MODEL, base_url=OLLAMA_BASE_URL, temperature=TEMPERATURE, reasoning=False, client_kwargs={'timeout': REQUEST_TIMEOUT_S})
openai_llm = ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE, reasoning_effort='none', timeout=REQUEST_TIMEOUT_S, max_retries=0)
planners = {
    'A. Local Qwen': (qwen_llm.with_structured_output(EntitySearchPlan, include_raw=True), QWEN_PROMPT),
    'B. OpenAI API': (openai_llm.with_structured_output(EntitySearchPlan, method='json_schema', include_raw=True), DETAILED_PROMPT),
}

RUN_CONFIG = {
    'run_at_utc': datetime.now(timezone.utc).isoformat(), 'local_model': LOCAL_MODEL, 'openai_model': OPENAI_MODEL,
    'temperature': TEMPERATURE, 'qwen_reasoning': False, 'openai_reasoning_effort': 'none', 'request_timeout_s': REQUEST_TIMEOUT_S,
    'prompt_version': PROMPT_VERSION, 'openai_input_usd_per_mtok': OPENAI_INPUT_USD_PER_MTOKEN,
    'openai_cached_input_usd_per_mtok': OPENAI_CACHED_INPUT_USD_PER_MTOKEN,
    'openai_output_usd_per_mtok': OPENAI_OUTPUT_USD_PER_MTOKEN,
    'langchain_version': version('langchain'), 'langchain_ollama_version': version('langchain-ollama'),
    'langchain_openai_version': version('langchain-openai'),
}
RUN_CONFIG

In [ ]:
def build_messages(system_prompt, question, raw_name):
    return [('system', system_prompt), ('human', f'사용자 질문: {question}\n원문 기업 표현: {raw_name}')]


def clean_terms(plan):
    if plan is None or plan.decision != 'candidate':
        return [], list(plan.search_terms) if plan is not None else []
    terms, rejected, seen = [], [], set()
    for term in plan.search_terms:
        term, key = term.strip(), normalize_name(term)
        if not term or len(term) > 80 or not key or key.isdigit() or key in seen:
            rejected.append(term)
            continue
        seen.add(key)
        terms.append(term)
    return terms[:2], rejected


def follows_output_contract(plan, raw_terms):
    if plan is None:
        return False
    if plan.decision == 'candidate':
        return 1 <= len(raw_terms) <= 2 and all(term.strip() for term in raw_terms)
    return plan.decision in {'ambiguous', 'unknown'} and not raw_terms


def usage_from_result(result):
    raw = result.get('raw') if isinstance(result, dict) else None
    usage = getattr(raw, 'usage_metadata', None) or {}
    metadata = getattr(raw, 'response_metadata', None) or {}
    token_usage = metadata.get('token_usage') or {}
    input_tokens = usage.get('input_tokens', token_usage.get('prompt_tokens'))
    output_tokens = usage.get('output_tokens', token_usage.get('completion_tokens'))
    total_tokens = usage.get('total_tokens', token_usage.get('total_tokens'))
    input_details = usage.get('input_token_details') or {}
    cached_input_tokens = input_details.get('cache_read', 0) or 0
    actual_model = metadata.get('model_name') or metadata.get('model') or getattr(raw, 'model_name', None)
    return input_tokens, output_tokens, total_tokens, cached_input_tokens, actual_model


def estimate_openai_cost(input_tokens, cached_input_tokens, output_tokens):
    if input_tokens is None or output_tokens is None:
        return None
    input_tokens = int(input_tokens)
    cached_input_tokens = min(max(int(cached_input_tokens or 0), 0), input_tokens)
    non_cached_input_tokens = input_tokens - cached_input_tokens
    return (
        non_cached_input_tokens * OPENAI_INPUT_USD_PER_MTOKEN
        + cached_input_tokens * OPENAI_CACHED_INPUT_USD_PER_MTOKEN
        + int(output_tokens) * OPENAI_OUTPUT_USD_PER_MTOKEN
    ) / 1_000_000


smoke_rows = []
for provider_name, (planner, system_prompt) in planners.items():
    print(f'[사전 호출] {provider_name} / 하닉 / 호출 시작', flush=True)
    started_at = time.perf_counter()
    try:
        result = planner.invoke(build_messages(system_prompt, '하닉 실적을 정리해줘.', '하닉'))
        input_tokens, output_tokens, total_tokens, cached_input_tokens, actual_model = usage_from_result(result)
        estimated_cost = estimate_openai_cost(input_tokens, cached_input_tokens, output_tokens) if provider_name == 'B. OpenAI API' else None
        smoke_rows.append({'제공처': provider_name, '입력 토큰': input_tokens, '캐시 입력 토큰': cached_input_tokens, '출력 토큰': output_tokens, '총 토큰': total_tokens, '응답 모델': actual_model, '사전 호출 추정 비용(USD)': estimated_cost})
        print(f'[사전 호출] 완료 / {(time.perf_counter() - started_at) * 1_000:,.0f}ms / parsed={result["parsed"]} / tokens=({input_tokens}, cache={cached_input_tokens}, {output_tokens}) / error={result["parsing_error"]}', flush=True)
    except Exception as exc:
        print(f'[사전 호출] 실패 / {(time.perf_counter() - started_at) * 1_000:,.0f}ms / {type(exc).__name__}: {exc}', flush=True)

display(pd.DataFrame(smoke_rows))

## 모델 후보 실행

LLM 호출 필요 사례마다 두 제공처를 모두 한 번씩 실행한다. 사례별로 A→B와 B→A를 교차하며, OpenAI API 호출의 token usage와 추정 비용을 함께 저장한다.

In [ ]:
PROVIDERS = list(planners)
llm_case_count = sum(needs_llm(case['원문 기업 표현']) for case in TEST_CASES)
call_total, call_number, rows = llm_case_count * len(PROVIDERS), 0, []
print(f'전체 평가 시작: LLM 호출 {call_total}회 ({llm_case_count}개 사례 × 제공처 {len(PROVIDERS)}개)', flush=True)

for case_number, case in enumerate(TEST_CASES, start=1):
    raw_name = case['원문 기업 표현']
    baseline = union_candidates(raw_name, 'raw')
    expected_codes = {target_records[name]['corp_code'] for name in case['기대 대상']}
    llm_called = needs_llm(raw_name)
    variants = {}
    order = PROVIDERS if case_number % 2 else list(reversed(PROVIDERS))

    for provider_name in order:
        planner, system_prompt = planners[provider_name]
        plan, error, raw_terms, terms, rejected, latency_ms = None, None, [], [], [], None
        llm_candidates, input_tokens, output_tokens, total_tokens = {}, None, None, None
        cached_input_tokens, actual_model = 0, None
        if llm_called:
            call_number += 1
            print(f'[{call_number}/{call_total}] {provider_name} / {raw_name!r} / 호출 시작', flush=True)
            started_at = time.perf_counter()
            try:
                result = planner.invoke(build_messages(system_prompt, case['질문'], raw_name))
                plan, error = result['parsed'], result['parsing_error']
                input_tokens, output_tokens, total_tokens, cached_input_tokens, actual_model = usage_from_result(result)
                raw_terms = list(plan.search_terms) if plan is not None else []
                terms, rejected = clean_terms(plan)
                llm_candidates = merge_candidates(*(union_candidates(term, 'llm') for term in terms))
            except Exception as exc:
                error = f'{type(exc).__name__}: {exc}'
            finally:
                latency_ms = (time.perf_counter() - started_at) * 1_000
                status = '완료' if error is None and plan is not None else '실패'
                print(f'[{call_number}/{call_total}] {status} / {latency_ms:,.0f}ms / tokens=({input_tokens}, cache={cached_input_tokens}, {output_tokens})', flush=True)
        variants[provider_name] = {
            'plan': plan, 'error': str(error) if error else None, 'raw_terms': raw_terms, 'terms': terms, 'rejected': rejected,
            'llm_candidates': llm_candidates, 'latency_ms': latency_ms, 'input_tokens': input_tokens,
            'output_tokens': output_tokens, 'total_tokens': total_tokens, 'cached_input_tokens': cached_input_tokens, 'actual_model': actual_model,
        }

    for provider_name in PROVIDERS:
        result = variants[provider_name]
        expanded = merge_candidates(baseline, result['llm_candidates'])
        plan, raw_terms = result['plan'], result['raw_terms']
        term_validity = [bool(term.strip()) and bool(exact_records(term.strip())) for term in raw_terms]
        estimated_cost = None
        if provider_name == 'B. OpenAI API' and result['input_tokens'] is not None and result['output_tokens'] is not None:
            estimated_cost = estimate_openai_cost(result['input_tokens'], result['cached_input_tokens'], result['output_tokens'])
        rows.append({
            **case, '제공처': provider_name, 'LLM 호출': llm_called, 'LLM 결정': plan.decision if plan else None,
            'LLM 원본 검색어': raw_terms, '실제 탐색 검색어': result['terms'], '거부된 검색어': result['rejected'],
            '구조화 출력 성공': None if not llm_called else result['error'] is None and plan is not None,
            '출력 계약 준수': None if not llm_called else follows_output_contract(plan, raw_terms),
            '원본 검색어 exact 유효 여부': term_validity,
            '기준선 정답 포함': bool(expected_codes & set(baseline)) if expected_codes else None,
            'LLM 정답 포함': bool(expected_codes & set(result['llm_candidates'])) if expected_codes else None,
            '확장 후 정답 포함': bool(expected_codes & set(expanded)) if expected_codes else None,
            'LLM 신규 기여': bool((expected_codes & set(result['llm_candidates'])) - set(baseline)) if expected_codes else None,
            '안전 보류': plan is not None and plan.decision in {'ambiguous', 'unknown'} and not raw_terms if case['LLM 기대 행동'] == 'abstain' else None,
            'LLM 호출 시간(ms)': result['latency_ms'], '입력 토큰': result['input_tokens'], '캐시 입력 토큰': result['cached_input_tokens'], '출력 토큰': result['output_tokens'],
            '총 토큰': result['total_tokens'], '응답 모델': result['actual_model'], '추정 비용(USD)': estimated_cost, '오류': result['error'],
        })

print(f'전체 평가 완료: 행 {len(rows)}개, LLM 호출 {call_number}회', flush=True)
len(rows)

In [ ]:
def p95(values):
    return sorted(values)[ceil(len(values) * 0.95) - 1] if values else None


summary_rows = []
for provider_name in PROVIDERS:
    provider_rows = [row for row in rows if row['제공처'] == provider_name]
    positive = [row for row in provider_rows if row['기대 대상']]
    candidate = [row for row in provider_rows if row['LLM 기대 행동'] == 'candidate' and row['LLM 호출']]
    abstain = [row for row in provider_rows if row['LLM 기대 행동'] == 'abstain']
    llm_rows = [row for row in provider_rows if row['LLM 호출']]
    validity = [value for row in llm_rows for value in row['원본 검색어 exact 유효 여부']]
    latencies = [row['LLM 호출 시간(ms)'] for row in llm_rows if row['LLM 호출 시간(ms)'] is not None]
    input_tokens = [row['입력 토큰'] for row in llm_rows if row['입력 토큰'] is not None]
    cached_input_tokens = [row['캐시 입력 토큰'] for row in llm_rows if row['캐시 입력 토큰'] is not None]
    output_tokens = [row['출력 토큰'] for row in llm_rows if row['출력 토큰'] is not None]
    costs = [row['추정 비용(USD)'] for row in llm_rows if row['추정 비용(USD)'] is not None]
    smoke_costs = [row['사전 호출 추정 비용(USD)'] for row in smoke_rows if row['제공처'] == provider_name and row['사전 호출 추정 비용(USD)'] is not None]
    response_models = sorted({row['응답 모델'] for row in llm_rows if row['응답 모델']})
    summary_rows.append({
        '제공처': provider_name, 'Candidate Recall': sum(row['확장 후 정답 포함'] for row in positive) / len(positive),
        'LLM Candidate Recall': sum(row['LLM 정답 포함'] for row in candidate) / len(candidate),
        'LLM Expansion Contribution': sum(row['LLM 신규 기여'] for row in positive) / len(positive),
        'Abstention Accuracy': sum(row['안전 보류'] for row in abstain) / len(abstain),
        'Structured-output Success Rate': sum(row['구조화 출력 성공'] for row in llm_rows) / len(llm_rows),
        'Output Contract Validity': sum(row['출력 계약 준수'] for row in llm_rows) / len(llm_rows),
        'OpenDART Exact Term Validity': sum(validity) / len(validity) if validity else None,
        'p50 ms': median(latencies) if latencies else None, 'p95 ms': p95(latencies),
        '입력 토큰 합계': sum(input_tokens) if input_tokens else None, '캐시 입력 토큰 합계': sum(cached_input_tokens) if cached_input_tokens else None, '출력 토큰 합계': sum(output_tokens) if output_tokens else None,
        '응답 모델': ', '.join(response_models) if response_models else None,
        '평가 호출 추정 비용(USD)': sum(costs) if costs else None,
        '사전 호출 추정 비용(USD)': sum(smoke_costs) if smoke_costs else None,
        '노트북 총 추정 비용(USD)': (sum(costs) if costs else 0) + (sum(smoke_costs) if smoke_costs else 0) if costs or smoke_costs else None,
    })

summary = pd.DataFrame(summary_rows).set_index('제공처')
display(summary)

In [ ]:
columns = [
    '제공처', '원문 기업 표현', 'LLM 기대 행동', 'LLM 결정', 'LLM 원본 검색어', '실제 탐색 검색어',
    'LLM 정답 포함', '확장 후 정답 포함', 'LLM 신규 기여', '안전 보류', '출력 계약 준수',
    '원본 검색어 exact 유효 여부', 'LLM 호출 시간(ms)', '입력 토큰', '캐시 입력 토큰', '출력 토큰', '응답 모델', '추정 비용(USD)', '오류',
]
display(pd.DataFrame(rows)[columns].sort_values(['원문 기업 표현', '제공처']))

## 채택 기준

- OpenAI API가 Local Qwen보다 Candidate Recall 또는 LLM Expansion Contribution을 높이고, 안전 보류·구조화 출력·출력 계약을 낮추지 않으면 API 후보를 조건부 채택한다.
- 안전 보류가 낮아지거나 무관한 실제 기업을 후보로 주입하면, Recall이 높아도 미채택한다.
- 품질 차이가 작으면 독립 holdout과 실제 비용을 함께 보고 최종 모델 선정은 보류한다.
- API token usage와 추정 비용은 `gpt-5.6-luna` 가격 기준의 참고값이다. 일반 입력·캐시 입력·출력 토큰을 분리하고, 사전 호출과 평가 호출 비용을 구분한다. 실제 청구 금액과 최신 가격은 OpenAI 콘솔·공식 가격 문서로 재확인한다.

실행 결과를 저장한 뒤 알려주면, 17번 HTML 보고서에 모델별 품질·지연시간·token·비용·채택 근거를 기록한다.